In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import ta

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

stocks = pd.read_parquet(
    INTERIM_DIR / "combined_nse_ohlcv_raw.parquet"
)

stocks = stocks.sort_values(["ticker", "Date"]).reset_index(drop=True)

nifty = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "NIFTY50.csv",
    parse_dates=["Date"]
).sort_values("Date")

print(stocks.shape, nifty.shape)

(121775, 10) (2821, 7)


In [2]:
%pip install ta

  Using cached ta-0.11.0-py3-none-any.whl
Note: you may need to restart the kernel to use updated packages.


In [2]:
def add_stock_features(group):
    group = group.sort_values("Date").copy()

    price = group["Adj Close"]

    # Returns and momentum
    group["past_return_5d"] = price.pct_change(5)
    group["past_return_10d"] = price.pct_change(10)
    group["past_return_20d"] = price.pct_change(20)

    # Moving-average distance
    group["ma20"] = price.rolling(20).mean()
    group["distance_ma20"] = price / group["ma20"] - 1

    # RSI
    group["rsi_14"] = ta.momentum.RSIIndicator(
        close=price,
        window=14
    ).rsi()

    # MACD histogram
    macd = ta.trend.MACD(
        close=price,
        window_slow=26,
        window_fast=12,
        window_sign=9
    )
    group["macd_histogram"] = macd.macd_diff()

    # ATR as percentage of price
    atr = ta.volatility.AverageTrueRange(
        high=group["High"],
        low=group["Low"],
        close=group["Close"],
        window=14
    ).average_true_range()
    group["atr_pct_14"] = atr / price

    # Bollinger features
    bb = ta.volatility.BollingerBands(
        close=price,
        window=20,
        window_dev=2
    )
    upper = bb.bollinger_hband()
    lower = bb.bollinger_lband()
    middle = bb.bollinger_mavg()

    group["bollinger_position"] = (
        (price - lower) / (upper - lower)
    )
    group["bollinger_width"] = (
        (upper - lower) / middle
    )

    # Rolling volatility
    daily_returns = price.pct_change()
    group["rolling_volatility_10d"] = (
        daily_returns.rolling(10).std()
    )

    # Volume features
    volume_ma20 = group["Volume"].rolling(20).mean()
    group["volume_ratio_20d"] = group["Volume"] / volume_ma20

    obv = ta.volume.OnBalanceVolumeIndicator(
        close=group["Close"],
        volume=group["Volume"]
    ).on_balance_volume()
    group["obv_change_10d"] = obv.pct_change(10)

    group["cmf_20"] = ta.volume.ChaikinMoneyFlowIndicator(
        high=group["High"],
        low=group["Low"],
        close=group["Close"],
        volume=group["Volume"],
        window=20
    ).chaikin_money_flow()

    # Mathematical features
    group["price_velocity_5d"] = price.pct_change(5)

    group["price_curvature_5d"] = (
        price.pct_change()
        .diff()
        .rolling(5)
        .mean()
    )

    return group

featured_stocks = (
    stocks.groupby("ticker", group_keys=False)
    .apply(add_stock_features)
    .reset_index(drop=True)
)

print(featured_stocks.shape)

(121775, 26)


/var/folders/dr/rhr144f11yg3pb3gl_2tg3040000gn/T/ipykernel_30158/1555540082.py:94: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_stock_features)


In [3]:
def rolling_entropy(values, bins=5):
    values = np.asarray(values)

    if np.isnan(values).any():
        return np.nan

    counts, _ = np.histogram(values, bins=bins)

    probabilities = counts / counts.sum()
    probabilities = probabilities[probabilities > 0]

    return -(probabilities * np.log(probabilities)).sum()


def rolling_spectral_norm(values):
    values = np.asarray(values)

    if np.isnan(values).any():
        return np.nan

    # Remove average return so this measures recent movement intensity,
    # not the direction of the average return.
    centered = values - values.mean()

    # Rolling return-path energy / spectral norm
    return np.sqrt(np.sum(centered ** 2))


def add_math_features(group):
    group = group.sort_values("Date").copy()

    daily_returns = group["Adj Close"].pct_change()

    group["return_entropy_10d"] = (
        daily_returns
        .rolling(10)
        .apply(rolling_entropy, raw=True)
    )

    group["spectral_norm_10d"] = (
        daily_returns
        .rolling(10)
        .apply(rolling_spectral_norm, raw=True)
    )

    return group


featured_stocks = (
    featured_stocks
    .groupby("ticker", group_keys=False)
    .apply(add_math_features)
    .reset_index(drop=True)
)

featured_stocks[
    [
        "ticker",
        "Date",
        "return_entropy_10d",
        "spectral_norm_10d",
        "price_curvature_5d"
    ]
].head(25)

/var/folders/dr/rhr144f11yg3pb3gl_2tg3040000gn/T/ipykernel_30158/666856695.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_math_features)


,ticker,Date,return_entropy_10d,spectral_norm_10d,price_curvature_5d
0,ADANIPORTS.NS,2015-01-01,NaN,NaN,NaN
1,ADANIPORTS.NS,2015-01-02,NaN,NaN,NaN
2,ADANIPORTS.NS,2015-01-05,NaN,NaN,NaN
3,ADANIPORTS.NS,2015-01-06,NaN,NaN,NaN
4,ADANIPORTS.NS,2015-01-07,NaN,NaN,NaN
5,ADANIPORTS.NS,2015-01-08,NaN,NaN,NaN
6,ADANIPORTS.NS,2015-01-09,NaN,NaN,-0.001907
7,ADANIPORTS.NS,2015-01-12,NaN,NaN,-0.004418
8,ADANIPORTS.NS,2015-01-13,NaN,NaN,-0.002114
9,ADANIPORTS.NS,2015-01-14,NaN,NaN,0.005574


In [4]:
nifty = nifty[["Date", "Adj Close"]].copy()

nifty = nifty.rename(
    columns={"Adj Close": "nifty_adj_close"}
)

nifty["nifty_return_5d"] = (
    nifty["nifty_adj_close"].pct_change(5)
)

nifty["nifty_return_20d"] = (
    nifty["nifty_adj_close"].pct_change(20)
)

nifty["nifty_volatility_10d"] = (
    nifty["nifty_adj_close"]
    .pct_change()
    .rolling(10)
    .std()
)

featured_stocks = featured_stocks.merge(
    nifty[
        [
            "Date",
            "nifty_return_5d",
            "nifty_return_20d",
            "nifty_volatility_10d"
        ]
    ],
    on="Date",
    how="left"
)

# Stock's 10-day performance relative to NIFTY's 10-day performance.
# This is more direct and easier to explain than the previous formula.
nifty["nifty_return_10d"] = (
    nifty["nifty_adj_close"].pct_change(10)
)

featured_stocks = featured_stocks.drop(
    columns=["nifty_return_10d"],
    errors="ignore"
)

featured_stocks = featured_stocks.merge(
    nifty[["Date", "nifty_return_10d"]],
    on="Date",
    how="left"
)

featured_stocks["relative_strength_10d"] = (
    featured_stocks["past_return_10d"]
    - featured_stocks["nifty_return_10d"]
)

print("Shape after adding NIFTY features:", featured_stocks.shape)

featured_stocks[
    [
        "Date",
        "ticker",
        "past_return_10d",
        "nifty_return_10d",
        "relative_strength_10d",
        "nifty_volatility_10d"
    ]
].head()

Shape after adding NIFTY features: (121775, 33)


,Date,ticker,past_return_10d,nifty_return_10d,relative_strength_10d,nifty_volatility_10d
0,2015-01-01,ADANIPORTS.NS,NaN,NaN,NaN,NaN
1,2015-01-02,ADANIPORTS.NS,NaN,NaN,NaN,NaN
2,2015-01-05,ADANIPORTS.NS,NaN,NaN,NaN,NaN
3,2015-01-06,ADANIPORTS.NS,NaN,NaN,NaN,NaN
4,2015-01-07,ADANIPORTS.NS,NaN,NaN,NaN,NaN


In [5]:
DECLINE_THRESHOLD = -0.05
REVERSAL_THRESHOLD = 0.03
FUTURE_WINDOW = 5

featured_stocks["future_return_5d"] = (
    featured_stocks
    .groupby("ticker")["Adj Close"]
    .shift(-FUTURE_WINDOW)
    / featured_stocks["Adj Close"]
    - 1
)

featured_stocks["candidate_decline"] = (
    featured_stocks["past_return_10d"] <= DECLINE_THRESHOLD
)

featured_stocks["reversal"] = np.where(
    featured_stocks["candidate_decline"]
    & (featured_stocks["future_return_5d"] >= REVERSAL_THRESHOLD),
    1,
    0
)

# Keep only candidate-decline days where the future 5-day return is known.
# This prevents the final 5 rows of each stock from being incorrectly labelled 0.
model_events = featured_stocks[
    featured_stocks["candidate_decline"]
    & featured_stocks["future_return_5d"].notna()
].copy()

model_events["reversal"] = model_events["reversal"].astype(int)

print("Candidate events with known future outcome:", len(model_events))
print("\nClass distribution:")
print(model_events["reversal"].value_counts())
print("\nClass percentage:")
print(
    model_events["reversal"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Candidate events with known future outcome: 15074

Class distribution:
reversal
0    11179
1     3895
Name: count, dtype: int64

Class percentage:
reversal
0    74.16
1    25.84
Name: proportion, dtype: float64


In [6]:
feature_columns = [
    # Momentum / trend
    "past_return_5d",
    "past_return_10d",
    "past_return_20d",
    "distance_ma20",
    "rsi_14",
    "macd_histogram",

    # Volatility
    "atr_pct_14",
    "bollinger_position",
    "bollinger_width",
    "rolling_volatility_10d",

    # Volume / money flow
    "volume_ratio_20d",
    "obv_change_10d",
    "cmf_20",

    # Mathematical features
    "price_velocity_5d",
    "price_curvature_5d",
    "return_entropy_10d",
    "spectral_norm_10d",

    # Market-context features
    "nifty_return_5d",
    "nifty_return_10d",
    "nifty_return_20d",
    "nifty_volatility_10d",
    "relative_strength_10d",
]

feature_missing = (
    model_events[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

feature_missing

macd_histogram            110
nifty_return_20d           33
relative_strength_10d      17
past_return_20d            17
distance_ma20              17
nifty_volatility_10d       17
bollinger_position         17
bollinger_width            17
volume_ratio_20d           17
nifty_return_10d           17
cmf_20                     17
nifty_return_5d            15
rsi_14                     11
return_entropy_10d          0
spectral_norm_10d           0
past_return_5d              0
price_curvature_5d          0
price_velocity_5d           0
past_return_10d             0
rolling_volatility_10d      0
atr_pct_14                  0
obv_change_10d              0
dtype: int64

In [7]:
# Keep metadata + target + all model features
metadata_columns = [
    "Date",
    "ticker",
    "company",
    "sector",
    "Adj Close",
    "future_return_5d",
    "reversal"
]

final_model_events = model_events[
    metadata_columns + feature_columns
].copy()

# Remove only rows where at least one feature is unavailable
final_model_events = final_model_events.dropna(
    subset=feature_columns
).copy()

final_model_events = final_model_events.sort_values(
    ["Date", "ticker"]
).reset_index(drop=True)

print("Rows before feature-null removal:", len(model_events))
print("Rows after feature-null removal:", len(final_model_events))
print("Rows removed:", len(model_events) - len(final_model_events))

print("\nFinal class distribution:")
print(final_model_events["reversal"].value_counts())

print("\nFinal class percentages:")
print(
    final_model_events["reversal"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nRemaining missing feature values:")
print(final_model_events[feature_columns].isna().sum().sum())

final_model_events.head()

Rows before feature-null removal: 15074
Rows after feature-null removal: 14949
Rows removed: 125

Final class distribution:
reversal
0    11109
1     3840
Name: count, dtype: int64

Final class percentages:
reversal
0    74.31
1    25.69
Name: proportion, dtype: float64

Remaining missing feature values:
0


,Date,ticker,company,sector,Adj Close,future_return_5d,reversal,past_return_5d,past_return_10d,past_return_20d,...,cmf_20,price_velocity_5d,price_curvature_5d,return_entropy_10d,spectral_norm_10d,nifty_return_5d,nifty_return_10d,nifty_return_20d,nifty_volatility_10d,relative_strength_10d
0,2015-02-19,HEROMOTOCO.NS,Hero MotoCorp,Automobile,1885.587280,-0.016023,0,-0.032106,-0.058155,-0.057872,...,-0.335243,-0.032106,-0.002351,1.470808,0.065176,0.031052,0.019671,0.022966,0.008014,-0.077825
1,2015-02-19,ONGC.NS,ONGC,Energy,125.908730,-0.042285,0,-0.033338,-0.095238,-0.048102,...,-0.087004,-0.033338,0.005981,1.504788,0.047839,0.031052,0.019671,0.022966,0.008014,-0.114909
2,2015-02-20,HEROMOTOCO.NS,Hero MotoCorp,Automobile,1881.306152,0.005152,0,-0.052947,-0.066168,-0.075213,...,-0.350975,-0.052947,-0.004391,1.504788,0.064090,0.014010,0.013993,0.011925,0.008454,-0.080161
3,2015-02-20,ONGC.NS,ONGC,Energy,124.870331,-0.017992,0,-0.045461,-0.072760,-0.044358,...,-0.091164,-0.045461,-0.002519,1.504788,0.041588,0.014010,0.013993,0.011925,0.008454,-0.086753
4,2015-02-20,RELIANCE.NS,Reliance Industries,Energy,185.972916,-0.020101,0,-0.039652,-0.054780,-0.037111,...,-0.064107,-0.039652,-0.008295,1.418484,0.048064,0.014010,0.013993,0.011925,0.008454,-0.068772


In [8]:
final_model_events.to_parquet(
    PROCESSED_DIR / "nse_reversal_model_dataset_v1.parquet",
    index=False
)

pd.Series(feature_columns, name="feature_name").to_csv(
    PROCESSED_DIR / "feature_list_v1.csv",
    index=False
)

print("Saved:")
print(PROCESSED_DIR / "nse_reversal_model_dataset_v1.parquet")
print(PROCESSED_DIR / "feature_list_v1.csv")

Saved:
/Users/kartiksinghai/Desktop/nse-stock-reversal-predictor/data/processed/nse_reversal_model_dataset_v1.parquet
/Users/kartiksinghai/Desktop/nse-stock-reversal-predictor/data/processed/feature_list_v1.csv
